# L73 · WER 评估 — 词错误率（插入 / 删除 / 替换）逐句分析

**目标**：系统评估微调（fine-tuning）后的语音识别模型——分解 WER 中 S/D/I 各自比例、分析逐句分布长尾、按领域（数字/专名/口音）定位薄弱点。

🔗 Aurora 连接：本节产出的 `analyze_errors()` 函数将来集成到 `aurora/speech/` 模块（当前为规划阶段，见 `src/aurora/speech/__init__.py`），为 `aurora.speech` 微调循环提供量化反馈信号。

← **上一课**　[L72 · Whisper 微调](L72_whisper_finetune.ipynb)

> 上节课学习了 **Whisper 微调**：LoRA 低秩注入 vs 全参数，中文 / 方言数据适配实战。  
> 本课将探讨 **WER 评估**。

## 学习目标

学完本节后，你应能：

1. **公式** — 写出 WER = (S+D+I)/N 并解释替换（S）、删除（D）、插入（I）三类错误的诊断含义；
2. **实现** — 用动态规划编辑对齐（DP back-trace）从词列表对中读出 S/D/I 计数，输出精确 WER；
3. **分析** — 从逐句 WER 分布中识别重尾样本，按领域（数字/专名/口音）定位薄弱点；
4. **对比** — 说明 jiwer 等工具库与手写实现的权衡（开箱即用 vs 可控细节），知道何时用哪种。


WER（词错误率）是 ASR 系统最常用的指标，但单一数字掩盖了错误的性质：一个漏词型模型和一个乱插词型模型 WER 可能相同，修复策略却截然不同。本节从 WER 拆解出发，分析逐句分布找出真正的失败样本，再按领域切分定位薄弱点，形成「测量→归因→改进」的完整闭环。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. WER 分解：S / D / I

WER 公式：

```
WER = (S + D + I) / N
```

N 是参考文本总词数，S = 替换（Substitution），D = 删除（Deletion），I = 插入（Insertion）。三类错误通过**动态规划（dynamic programming，DP）编辑距离回溯**读出：回溯时走对角线是替换/匹配，向上走是插入（hypothesis 多一词），向左走是删除（reference 多一词）。

典型分布：对话场景 S 占主导（约 60-70%）；口音重时 D 升高（漏词）；解码器过于激进时 I 升高幻觉词（hallucination）。三项比率决定修复方向。

In [ ]:
# 演示：手工对齐一对句子，观察 S/D/I 操作
def _edit_ops(hyp, ref):
    H, R = len(hyp), len(ref)
    dp = [[0]*(R+1) for _ in range(H+1)]
    for i in range(H+1): dp[i][0] = i
    for j in range(R+1): dp[0][j] = j
    for i in range(1, H+1):
        for j in range(1, R+1):
            if hyp[i-1] == ref[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    ops = []
    i, j = H, R
    while i > 0 or j > 0:
        if i > 0 and j > 0 and hyp[i-1] == ref[j-1]:
            ops.append('C'); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append('S'); i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append('I'); i -= 1   # hyp 多了一词
        else:
            ops.append('D'); j -= 1   # ref 多了一词
    ops.reverse()
    return ops

hyp = ['the', 'cat', 'sat']
ref = ['the', 'big', 'cat', 'sat', 'down']
ops = _edit_ops(hyp, ref)
counts = {op: ops.count(op) for op in 'SDIC'}
print('对齐操作序列:', ops)
print(f"S={counts['S']}  D={counts['D']}  I={counts['I']}  C={counts['C']}")
print(f"WER = ({counts['S']}+{counts['D']}+{counts['I']}) / {len(ref)} "
      f"= {(counts['S']+counts['D']+counts['I'])/len(ref):.2f}")

## 2. 逐句 WER 分布

WER 在句子粒度呈**重尾分布（heavy-tail distribution）**：大多数句子 WER=0（完全识别正确），少数句子 WER 极高甚至超过 100%。超过 100% 发生在插入词数 I 超过参考词数 N 时：

```
WER_i > 1.0  ↔  I_i > N_i  （幻觉词比真实词还多）
```

分布均值被长尾拉高，**中位数**更能描述典型表现。重点关注 WER > 50% 的样本——它们通常对应噪声环境、非母语口音、领域专名或超长句子，是微调数据增强的优先候选。

In [ ]:
# 演示：模拟一批 WER，观察分布与长尾
np.random.seed(42)
wers = np.concatenate([
    np.random.beta(1, 8, 80) * 0.4,   # 低错误率主体
    np.random.uniform(0.5, 1.5, 15),   # 高错误长尾
    np.random.uniform(1.0, 2.0, 5),    # WER > 1（插入过多）
])

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(wers, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(wers), color='red', linestyle='--',
                label=f'均值 {np.mean(wers):.2f}')
axes[0].axvline(np.median(wers), color='orange', linestyle='--',
                label=f'中位数 {np.median(wers):.2f}')
axes[0].set_xlabel('WER'); axes[0].set_ylabel('句子数')
axes[0].set_title('逐句 WER 分布（模拟）')
axes[0].legend()

axes[1].hist(wers, bins=30, color='steelblue', edgecolor='white',
             cumulative=True, density=True)
axes[1].axhline(0.8, color='green', linestyle=':', label='80% 分位')
axes[1].set_xlabel('WER'); axes[1].set_ylabel('累积比例')
axes[1].set_title('累积分布 CDF')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'WER > 50% 的句子：{(wers > 0.5).sum()} / {len(wers)}')
print(f'WER > 100% 的句子：{(wers > 1.0).sum()} / {len(wers)}')

## 3. 按领域分析：数字 / 专名 / 口音

整体 WER 是各子集的加权平均，不同领域错误类型差异明显：

- **数字**：写法不一致（"一九九七" vs "1997"）导致 I/S 虚高，语义正确但 WER 偏大
- **专名/术语**：训练覆盖不足，S（替换）占主导
- **口音/方言**：音素识别失败导致漏词，D（删除）升高

```
诊断流程：
  按领域分桶 → 各桶独立计算 (S_rate, D_rate, I_rate)
  → 错误类型分布差异 → 指向具体修复方向
```

专名 → 扩充词表/提示；口音 → 数据增强；数字 → 后处理规则。

In [ ]:
# 演示：三个领域的 S/D/I 对比柱状图
domains = ['通用句子', '含数字', '含专名']
stats = {
    '通用句子': {'S': 0.07, 'D': 0.03, 'I': 0.02},
    '含数字':   {'S': 0.04, 'D': 0.02, 'I': 0.12},  # 幻觉词多，I 高
    '含专名':   {'S': 0.18, 'D': 0.05, 'I': 0.03},  # 替换多，S 高
}

x = np.arange(len(domains))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
for i, (op, color, label) in enumerate([
    ('S', '#E07B54', 'S 替换'),
    ('D', '#5B8DB8', 'D 删除'),
    ('I', '#6DAC6D', 'I 插入'),
]):
    vals = [stats[d][op] for d in domains]
    ax.bar(x + i*width, vals, width, label=label, color=color)
ax.set_xticks(x + width)
ax.set_xticklabels(domains)
ax.set_ylabel('错误率')
ax.set_title('各领域 S / D / I 错误率对比')
ax.legend()
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `analyze_errors(hypotheses, references)`

**推理路线**：
1. 对每对 `(hyp, ref)`（均为词列表）执行动态规划编辑对齐（alignment），回溯读出 S / D / I 计数
2. 累加所有句子的 S、D、I、N（参考词总数），计算全局 WER 及各类比率：
   `S_rate = total_S / total_N`，以此类推
3. 对每个句子计算逐句 WER，取最高的 5 个作为 `worst_examples`（含句子索引和 WER 值）

**参考输入输出**：
```python
# 约 100 句、1000 词的评估集，典型返回：
{
    'wer':    0.12,   # (S+D+I) / N = (70+30+20) / 1000
    'S_rate': 0.07,   # 70 / 1000
    'D_rate': 0.03,   # 30 / 1000
    'I_rate': 0.02,   # 20 / 1000
    'worst_examples': [(23, 1.5), (67, 1.2), (11, 0.8), (44, 0.7), (89, 0.6)]
}
```

> 💡 需要提示？参考 Cell 5 中 `_edit_ops` 的回溯逻辑，对各操作类型计数即可得到 (S, D, I)，无需重写完整 DP。
>
> **L67 前置**：本节 DP 实现是 L67 编辑距离的直接延伸；若需复习回溯逻辑，请参考 `L67_edit_distance.ipynb`。


In [ ]:
def analyze_errors(hypotheses, references):
    # hypotheses: list[list[str]]  模型识别输出（词列表的列表）
    # references:  list[list[str]]  参考文本（词列表的列表）
    # 返回: dict，包含 wer / S_rate / D_rate / I_rate / worst_examples

    def edit_counts(hyp, ref):
        # ✏️ TODO: 动态规划构建 dp 矩阵，然后回溯，返回 (S, D, I)
        raise NotImplementedError("TODO: 实现动态规划编辑对齐，返回 (S, D, I) 三元组")

    total_S = total_D = total_I = total_N = 0
    per_wer = []
    for hyp, ref in zip(hypotheses, references):
        # ✏️ TODO: 调用 edit_counts，累加 S/D/I/N，记录逐句 WER
        raise NotImplementedError("TODO: 调用 edit_counts，累加 S/D/I/N，计算逐句 WER")

    # ✏️ TODO: 从 per_wer 中找出 WER 最高的 5 个样本（含索引）
    worst = []

    return {
        'wer': (total_S + total_D + total_I) / max(total_N, 1),
        'S_rate': total_S / max(total_N, 1),
        'D_rate': total_D / max(total_N, 1),
        'I_rate': total_I / max(total_N, 1),
        'worst_examples': worst,
    }


In [ ]:
# 检查：已知答案验证
# hyp[0] 缺一个词（D=1，N=3）；hyp[1] 多一个词（I=1，N=2）
# 总计：S=0, D=1, I=1, N=5  →  WER = 2/5 = 0.40
# 精确期望：S_rate=0/5=0.0, D_rate=1/5=0.2, I_rate=1/5=0.2
hyps = [['the', 'cat'], ['hello', 'world', 'today']]
refs = [['the', 'big', 'cat'], ['hello', 'world']]

try:
    result = analyze_errors(hyps, refs)
except NotImplementedError as _e:
    print(f"⬜ analyze_errors 未实现：{_e}")
    result = None

if result is not None:
    assert isinstance(result, dict), '返回值应为 dict'
    assert {'wer','S_rate','D_rate','I_rate','worst_examples'} <= set(result.keys()), '缺少必要 key'

    # 精确 WER 断言（Fix C2）
    assert abs(result['wer'] - 0.40) < 1e-9, (
        f"WER 期望 0.400，实际 {result['wer']:.6f}；"
        f"请检查 S/D/I 计数是否正确（S=0,D=1,I=1,N=5）"
    )
    # 精确各分类比率断言
    assert abs(result['S_rate'] - 0.0) < 1e-9, f"S_rate 期望 0.0，实际 {result['S_rate']:.6f}"
    assert abs(result['D_rate'] - 0.2) < 1e-9, f"D_rate 期望 0.200，实际 {result['D_rate']:.6f}"
    assert abs(result['I_rate'] - 0.2) < 1e-9, f"I_rate 期望 0.200，实际 {result['I_rate']:.6f}"

    rate_sum = result['S_rate'] + result['D_rate'] + result['I_rate']
    assert abs(rate_sum - result['wer']) < 1e-9, (
        f'S+D+I rate 之和应等于 WER，得 {rate_sum:.4f} vs {result["wer"]:.4f}'
    )
    assert result['wer'] >= 0.0, f"WER 不能为负，实际 {result['wer']:.3f}"
    assert len(result['worst_examples']) <= 5, 'worst_examples 最多 5 条'

    print('✅ analyze_errors 全部验证通过')
    print(f"  WER     = {result['wer']:.3f}  ✓ 精确匹配 0.400")
    print(f"  S_rate  = {result['S_rate']:.3f}  D_rate = {result['D_rate']:.3f}  I_rate = {result['I_rate']:.3f}")
    print(f"  worst_examples: {result['worst_examples']}")


## 延伸：jiwer 库对比

手写 `analyze_errors()` 是本课重点（理解 DP 回溯机制）。生产环境中常用 [jiwer](https://github.com/jitsi/jiwer) 库作为快速参照：

```python
# pip install jiwer  （可选，不影响本课评分）
import jiwer

reference = "the big cat"
hypothesis = "the cat"

measures = jiwer.compute_measures(reference, hypothesis)
# 输出: {'wer': 0.333, 'mer': 0.333, 'wil': 0.333, ...}
# 注意：jiwer 对单句用 N=3，WER=D/N=1/3≈0.333
```

**对比权衡**：

| 维度 | 手写实现 | jiwer |
|------|----------|-------|
| 控制力 | 完整（可定制对齐规则） | 有限（固定实现） |
| 速度 | O(H·R) per pair | 同级但有 C 加速 |
| 可解释性 | 直接读回 S/D/I | 需解析输出字段 |
| 研究场景 | ✅ 推荐 | 快速验证用 |

> Aurora 项目遵循「不用黑盒」原则——`analyze_errors()` 的手写实现是核心交付物，jiwer 仅作参照。


## 5. 参数实验：WER 分布直方图与基线对比

**实验参数**：
- `bins`：直方图分桶数，10 看粗粒度形态，50 看长尾细节
- `finetuned_wers`：微调模型的逐句 WER 列表（来自 `analyze_errors` 遍历 per_wer）
- `baseline_wers`：未微调 Whisper-small 在同一测试集的逐句 WER 列表

**预期现象**：
- 微调后分布整体左移，低 WER 区间（0–0.1）堆积更多句子
- WER > 1.0 的极端样本减少，说明幻觉词被压制
- 若均值下降幅度小于中位数下降幅度，说明长尾仍有顽固失败样本，需针对性增强

In [ ]:
# 参数实验：微调前后 WER 分布对比（实际使用时替换为真实评估结果）
np.random.seed(0)

baseline_wers = np.concatenate([
    np.random.beta(2, 5, 70) * 0.8,
    np.random.uniform(0.6, 2.0, 30),
])
finetuned_wers = np.concatenate([
    np.random.beta(1, 10, 85) * 0.4,
    np.random.uniform(0.4, 1.2, 15),
])

bins = 30  # ✏️ 调整 bins 观察粗细粒度

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(baseline_wers, bins=bins, alpha=0.6, color='#E07B54',
        label=f'Whisper-small 基线  均值={np.mean(baseline_wers):.2f}  '
              f'中位数={np.median(baseline_wers):.2f}')
ax.hist(finetuned_wers, bins=bins, alpha=0.6, color='#5B8DB8',
        label=f'微调后            均值={np.mean(finetuned_wers):.2f}  '
              f'中位数={np.median(finetuned_wers):.2f}')
ax.axvline(1.0, color='black', linestyle=':', linewidth=1,
           label='WER = 1.0 分界（幻觉词超过参考词）')
ax.set_xlabel('逐句 WER')
ax.set_ylabel('句子数')
ax.set_title('微调前后 WER 分布对比')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'基线   WER>1.0 句子数：{(baseline_wers > 1.0).sum()}')
print(f'微调后 WER>1.0 句子数：{(finetuned_wers > 1.0).sum()}')

## 本课收束

本节实现了 `analyze_errors()`，输出全局 WER 及 `S_rate / D_rate / I_rate` 三项分类比率，并定位 `worst_examples` 中 WER 最高的 5 个句子。该函数将来集成到 `aurora/speech/` 模块（当前为规划阶段），作为 `aurora.speech` 微调循环的量化反馈信号。下一节（L74）将深入 S/D/I 错误模式：构建替换混淆矩阵（confusion matrix），统计最常见的替换/删除/插入词，将 WER 从单一数字转化为可操作的词汇错误列表。

---

→ **下一课**　[L74 · ASR 错误分析](L74_asr_error_analysis.ipynb)

> 下节课将学习 **ASR 错误分析**：替换/删除/插入模式，从 WER 到可改进方向。